# Project 2: Learning Word Embeddings with Feed Forward Neural Networks (PyTorch)

## Overview

In the previous project, we used Word2Vec to learn word embeddings from movie overviews and built a content-based movie recommendation system. While the model produced meaningful embeddings, the learning process remained largely hidden behind a library implementation.

In this project, we will build the core idea ourselves using PyTorch.

The objective is to understand how a feed forward neural network can learn semantic representations of words by predicting their surrounding context. Instead of treating embeddings as pre-generated vectors, we will learn how they emerge naturally as trainable parameters within a neural network.

By training the model on movie overviews, we will create our own embedding space and use the learned embeddings to build a movie recommendation system, allowing us to directly compare our implementation against the Word2Vec baseline from Project 1.

---

## Learning Objectives

By the end of this project, we will understand:

- How text is converted into numerical representations.
- How vocabularies and word-to-index mappings are constructed.
- How training examples are generated from context windows.
- How embedding layers work internally.
- How feed forward neural networks process inputs.
- How loss functions and optimization drive learning.
- How embeddings emerge as a by-product of context prediction.
- How to train neural networks using PyTorch.

---

## Project Pipeline

Raw Movie Overviews

↓

Text Cleaning & Tokenization

↓

Vocabulary Construction

↓

Word → ID Mapping

↓

Generate (Center Word, Context Word) Training Pairs

↓

Feed Forward Neural Network

↓

Learn Word Embeddings

↓

Create Movie Embeddings

↓

Cosine Similarity Search

↓

Movie Recommendations

↓

Genre-Based Evaluation

---

## Neural Network Architecture

Input Word ID

↓

Embedding Layer

↓

Dense (Linear) Layer

↓

Vocabulary Scores

↓

Cross Entropy Loss

↓

Backpropagation

↓

Updated Embeddings

The network's primary task is to predict nearby words. The learned embeddings are not the direct output of the model but rather an internal representation that develops during training.

---

## Evaluation Strategy

To evaluate the quality of the learned embeddings, we will reuse the genre-overlap framework developed in Project 1.

For each movie:

1. Generate recommendations using embedding similarity.
2. Retrieve the top-k most similar movies.
3. Measure how many recommendations share at least one genre with the query movie.
4. Compute the overall genre match rate.

Baseline from Project 1:

Word2Vec Genre Match Rate = 58.34%

The goal is to compare our neural network implementation against this benchmark and understand the strengths and limitations of learned embeddings.

---

## Expected Outcomes

At the completion of this project, we will have:

- Built a feed forward neural network in PyTorch.
- Trained our own word embeddings from scratch.
- Understood how embeddings are learned rather than simply used.
- Applied those embeddings to a real recommendation problem.
- Established a foundation for modern NLP models such as BERT, Sentence Transformers, and Large Language Models.

This project serves as the bridge between traditional word embedding techniques and modern deep learning-based language representations.

In [1]:
import pandas as pd
import numpy as np
import re
import os

In [2]:
path = r"C:\Users\ishik\.cache\kagglehub\datasets\rounakbanik\the-movies-dataset\versions\7"

In [3]:
files = os.listdir(path)

files

['credits.csv',
 'keywords.csv',
 'links.csv',
 'links_small.csv',
 'movies_metadata.csv',
 'ratings.csv',
 'ratings_small.csv']

In [4]:
movies_df = pd.read_csv(
    path + "/movies_metadata.csv",
    low_memory=False
)

In [5]:
movies_df.shape

(45466, 24)

In [6]:
movies_df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [7]:
movies_df = movies_df[
    ["title", "overview", "genres"]
].copy()

In [8]:
movies_df.head()

,title,overview,genres
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]"


In [9]:
movies_df.shape

(45466, 3)

In [10]:
movies_df = movies_df.dropna(
    subset=["overview"]
).reset_index(drop=True)

In [11]:
movies_df.shape

(44512, 3)

In [12]:
def clean_text(text):
    text=text.lower()
    text=re.sub(r"[^A-Za-z\s]","",text)
    tokens=text.split()
    return tokens

In [13]:
movies_df["overview"].iloc[0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."

In [14]:
clean_text(
    movies_df["overview"].iloc[0]
)

['led',
 'by',
 'woody',
 'andys',
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 'andys',
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'the',
 'scene',
 'afraid',
 'of',
 'losing',
 'his',
 'place',
 'in',
 'andys',
 'heart',
 'woody',
 'plots',
 'against',
 'buzz',
 'but',
 'when',
 'circumstances',
 'separate',
 'buzz',
 'and',
 'woody',
 'from',
 'their',
 'owner',
 'the',
 'duo',
 'eventually',
 'learns',
 'to',
 'put',
 'aside',
 'their',
 'differences']

In [15]:
corpus = movies_df["overview"].apply(clean_text)

In [16]:
corpus

0        [led, by, woody, andys, toys, live, happily, i...
1        [when, siblings, judy, and, peter, discover, a...
2        [a, family, wedding, reignites, the, ancient, ...
3        [cheated, on, mistreated, and, stepped, on, th...
4        [just, when, george, banks, has, recovered, fr...
                               ...                        
44507    [rising, and, falling, between, a, man, and, w...
44508    [an, artist, struggles, to, finish, his, work,...
44509    [when, one, of, her, hits, goes, wrong, a, pro...
44510    [in, a, small, town, live, two, brothers, one,...
44511    [years, after, decriminalisation, of, homosexu...
Name: overview, Length: 44512, dtype: object

In [17]:
type(corpus)

pandas.Series

In [18]:
type(corpus.iloc[0])

list

In [19]:
len(corpus)

44512

In [20]:
from collections import Counter


In [21]:
word_counts = Counter()

for sentence in corpus:
    word_counts.update(sentence)

In [22]:
len(word_counts)

88932

In [23]:
word_counts.most_common(20)

[('the', 138476),
 ('a', 99063),
 ('and', 75413),
 ('to', 73447),
 ('of', 69725),
 ('in', 48236),
 ('is', 36554),
 ('his', 36213),
 ('with', 23934),
 ('her', 21523),
 ('he', 20501),
 ('for', 18367),
 ('on', 17472),
 ('an', 16811),
 ('by', 15806),
 ('that', 15404),
 ('as', 14756),
 ('who', 14408),
 ('their', 13175),
 ('from', 12542)]

In [24]:
rare_words = sum(
    1
    for count in word_counts.values()
    if count < 5
)

rare_words

65726

In [25]:
vocab_words = [
    word
    for word, count in word_counts.items()
    if count >= 5
]

len(vocab_words)

23206

In [26]:
word_to_id = {
    word: idx
    for idx, word in enumerate(vocab_words)
}

In [27]:
id_to_word = {
    idx: word
    for word, idx in word_to_id.items()
}

In [28]:
list(word_to_id.items())[:10]

[('led', 0),
 ('by', 1),
 ('woody', 2),
 ('andys', 3),
 ('toys', 4),
 ('live', 5),
 ('happily', 6),
 ('in', 7),
 ('his', 8),
 ('room', 9)]



## Dataset Evolution Till Now

### Stage 1: Original Dataset

We start with the TMDB Movies Dataset.

| title | overview | genres |
|---------|---------|---------|
| Troy | In year 1250 B.C. during the late Bronze Age... | Adventure, Drama, War |
| Toy Story | Led by Woody, Andy's toys live happily... | Animation, Comedy, Family |
| The Dark Knight | Batman raises the stakes in his war on crime... | Action, Crime, Drama |

At this stage, each movie is represented using human-readable text.

---

### Stage 2: Keep Relevant Information

For our recommendation system, we keep:

| title | overview | genres |
|---------|---------|---------|
| Troy | In year 1250 B.C. during the late Bronze Age... | Adventure, Drama, War |

The overview contains the language from which we want to learn meaning.

The title will later be used for recommendations.

The genres will later be used for evaluation.

---

### Stage 3: Text Cleaning

Raw overview:

> In year 1250 B.C. during the late Bronze Age...

After cleaning:

> in year bc during the late bronze age

Punctuation, numbers and capitalization are removed to create a more consistent vocabulary.

---

### Stage 4: Tokenization

The cleaned sentence is converted into individual words.

| Before | After |
|----------|----------|
| in year bc during the late bronze age | ['in', 'year', 'bc', 'during', 'the', 'late', 'bronze', 'age'] |

The model learns from words, not entire sentences.

---

### Stage 5: Corpus Creation

Each movie overview becomes a list of tokens.

| Movie | Tokenized Overview |
|---------|---------|
| Troy | ['in', 'year', 'bc', 'during', 'the', 'late', 'bronze', 'age'] |
| Toy Story | ['led', 'by', 'woody', 'andys', 'toys'] |
| The Dark Knight | ['batman', 'raises', 'the', 'stakes', 'in', 'his', 'war', 'on', 'crime'] |

The collection of all tokenized overviews is called the **corpus**.

---

### Stage 6: Vocabulary Extraction

From the entire corpus:

| Word | Frequency |
|---------|---------|
| the | 138,476 |
| a | 99,063 |
| and | 75,413 |
| war | 2,118 |
| love | 3,421 |

Statistics:

- Initial unique words: ~88,000
- Rare words removed: ~65,000
- Final vocabulary: 23,206 words

The vocabulary represents all words that the model will learn embeddings for.

---

### Stage 7: Word Indexing

Words are converted into unique identifiers.

| Word | ID |
|---------|---------|
| the | 0 |
| a | 1 |
| and | 2 |
| war | 523 |
| love | 811 |

At this point, every word in our vocabulary has a fixed numerical identity.

---

## Current State

Movie Overview

↓

Clean Text

↓

Tokens

↓

Corpus

↓

Vocabulary

↓

Word IDs ✅

---

## Next Stage

Convert each movie overview into a sequence of word IDs.

Example:

| Tokens | Numerical Representation |
|----------|----------|
| ['war', 'begins', 'today'] | [523, 1741, 98] |

This transforms human language into numerical data that can be processed by a neural network.

In [29]:
corpus.iloc[0]

['led',
 'by',
 'woody',
 'andys',
 'toys',
 'live',
 'happily',
 'in',
 'his',
 'room',
 'until',
 'andys',
 'birthday',
 'brings',
 'buzz',
 'lightyear',
 'onto',
 'the',
 'scene',
 'afraid',
 'of',
 'losing',
 'his',
 'place',
 'in',
 'andys',
 'heart',
 'woody',
 'plots',
 'against',
 'buzz',
 'but',
 'when',
 'circumstances',
 'separate',
 'buzz',
 'and',
 'woody',
 'from',
 'their',
 'owner',
 'the',
 'duo',
 'eventually',
 'learns',
 'to',
 'put',
 'aside',
 'their',
 'differences']

In [30]:
[word_to_id[word]
 for word in corpus.iloc[0]
 if word in word_to_id]

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 3,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 8,
 20,
 7,
 3,
 21,
 2,
 22,
 23,
 13,
 24,
 25,
 26,
 27,
 13,
 28,
 2,
 29,
 30,
 31,
 15,
 32,
 33,
 34,
 35,
 36,
 37,
 30,
 38]

In [31]:
sentence_ids = [
    word_to_id[word]
    for word in corpus.iloc[0]
    if word in word_to_id
]

In [32]:
sentence_ids[:20]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3, 11, 12, 13, 14, 15, 16, 17, 18]

In [33]:
def generate_pairs(sentence_ids):

    pairs = []

    for i in range(len(sentence_ids)):

        center = sentence_ids[i]

        if i > 0:
            pairs.append(
                (center, sentence_ids[i-1])
            )

        if i < len(sentence_ids)-1:
            pairs.append(
                (center, sentence_ids[i+1])
            )

    return pairs

In [34]:
pairs = generate_pairs(sentence_ids)

pairs[:10]

[(0, 1),
 (1, 0),
 (1, 2),
 (2, 1),
 (2, 3),
 (3, 2),
 (3, 4),
 (4, 3),
 (4, 5),
 (5, 4)]

In [35]:
corpus_ids = []

for sentence in corpus:

    sentence_ids = [
        word_to_id[word]
        for word in sentence
        if word in word_to_id
    ]

    corpus_ids.append(sentence_ids)

In [36]:
len(corpus_ids)

44512

In [37]:
corpus_ids[0][:20]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 3, 11, 12, 13, 14, 15, 16, 17, 18]

In [38]:
all_pairs = []

for sentence_ids in corpus_ids:

    pairs = generate_pairs(sentence_ids)

    all_pairs.extend(pairs)

In [39]:
len(all_pairs)

4579326

In [40]:
all_pairs[:10]

[(0, 1),
 (1, 0),
 (1, 2),
 (2, 1),
 (2, 3),
 (3, 2),
 (3, 4),
 (4, 3),
 (4, 5),
 (5, 4)]

45k Movie Overviews
↓
Tokenization
↓
Vocabulary (23k words)
↓
Word IDs
↓
Corpus IDs
↓
4.58 Million (Center, Context) Pairs ✅

In [41]:
# all_pairs -> pytorch tensors, next task

In [42]:
# First let's separate inputs and targets
inputs = [pair[0] for pair in all_pairs]
targets = [pair[1] for pair in all_pairs]

In [43]:
inputs[:10]

[0, 1, 1, 2, 2, 3, 3, 4, 4, 5]

In [44]:
targets[:10]

[1, 0, 2, 1, 3, 2, 4, 3, 5, 4]

In [45]:
#Convert to tensors
import torch

X = torch.tensor(inputs)
y = torch.tensor(targets)

In [46]:
type(X)

torch.Tensor

In [47]:
type(y)

torch.Tensor

In [48]:
X.shape
y.shape

torch.Size([4579326])

In [49]:
vocab_size = len(word_to_id)

vocab_size

23206

In [50]:
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=100
)

In [51]:
embedding.weight.shape

torch.Size([23206, 100])

## Embedding Layer (Where We Are Now)

We have:

- Vocabulary Size = 23,206
- Embedding Dimension = 100

```python
embedding = nn.Embedding(
    num_embeddings=23206,
    embedding_dim=100
)
```

This creates a trainable matrix:

```text
Embedding Matrix

23206 × 100
```

Each row corresponds to one word:

```text
Word ID 0  → [x1, x2, ..., x100]
Word ID 1  → [x1, x2, ..., x100]
...
Word ID 523 (Troll2) → [x1, x2, ..., x100]
```

Initially, these values are random.

When a word ID is given as input:

```text
523
↓
Embedding Lookup
↓
embedding.weight[523]
↓
100-dimensional vector
```

The embedding layer performs **lookup only**.

```text
Word ID
↓
Embedding Matrix
↓
Embedding Vector
```

No activation function is applied.

No linear transformation is performed.

It is effectively an identity operation after fetching the corresponding row.

These embedding vectors are the representations that will ultimately be learned during training.

Word ID
↓
Embedding (100)
↓
Linear (100 → 128)
↓
ReLU
↓
Linear (128 → 23206)
↓
Softmax
↓
Predicted Context Word

In [52]:
## Building First Linear Layer
linear1 = nn.Linear(
    in_features=100,
    out_features=128
)
#128 is an hp and is a deliberate choice

In [53]:
linear1.weight.shape

torch.Size([128, 100])

In [54]:
linear1.bias.shape

torch.Size([128])

In [55]:
sample_word = torch.tensor([523])

In [56]:
embedding_output = embedding(sample_word)

embedding_output.shape

torch.Size([1, 100])

In [57]:
embedding_output #100 nos

tensor([[ 1.3168,  1.0903, -0.2130, -0.3454, -0.0685,  1.5591, -0.3386, -1.1753,
         -0.7679, -0.1584, -1.3154, -0.1543, -2.2018,  1.4015,  0.2776,  1.3010,
          1.4262,  0.0117,  0.4920,  3.6904,  0.2429,  1.4279,  0.9613, -0.0930,
         -0.4764, -0.3802,  0.6134,  0.2426,  1.2339,  1.8666,  0.6924, -0.6090,
          0.3521, -0.3531, -1.4281, -0.8402, -0.4721,  1.7510, -0.3820,  1.2097,
         -1.0580, -0.3941,  0.6916,  0.7253, -0.1488,  0.4936,  0.2965, -0.7040,
         -0.2586, -0.1181, -1.5858, -0.3033,  0.5595, -0.2110, -0.4496, -0.5765,
          0.6889, -0.1503,  0.1347, -0.0233, -0.2817,  1.0741,  1.4705, -1.5231,
          2.2280, -0.3828, -0.1793,  0.6969, -0.1899,  0.7164,  0.9919,  0.5422,
          0.9398, -0.1060, -1.2063,  0.0955,  0.1419, -0.4755, -0.4374,  0.6195,
         -0.9427,  1.4373,  0.9678,  0.8449,  0.2961,  2.8777, -0.5586, -0.7421,
         -1.0403, -0.8685, -1.2591,  1.6048, -1.6954,  0.8419,  0.0460, -0.1660,
          0.9803, -0.2007, -

In [58]:
hidden = linear1(embedding_output)

In [59]:
hidden #128 nos

tensor([[-4.2092e-01,  6.8060e-01, -4.2309e-01, -6.1201e-01, -2.4358e-01,
         -1.0026e-01, -2.2934e-01,  3.9914e-01,  5.0401e-01,  2.0539e-01,
         -9.0368e-02, -3.2827e-02,  7.4012e-02, -2.5368e-01,  6.4233e-01,
         -3.4970e-01, -2.2482e-02, -8.9981e-01, -9.0630e-01,  3.5352e-02,
          5.3261e-01,  1.6032e-01,  8.9690e-03,  2.4817e-01,  2.1135e-01,
          1.5865e+00, -7.3539e-01,  1.6425e-01, -5.9507e-01,  1.0822e-01,
         -6.3679e-02, -3.5508e-01, -5.2413e-02, -1.5351e-01,  1.3434e-01,
         -7.2280e-01, -3.3937e-01, -6.2873e-01,  3.5410e-01, -8.2358e-01,
         -3.8789e-02,  7.9332e-01,  8.0955e-01,  5.2425e-01, -8.2976e-01,
          4.7808e-01, -7.6136e-01,  1.3205e-01, -2.9016e-01,  4.5045e-01,
         -8.7151e-01, -5.3666e-01, -5.3680e-01,  5.6927e-01,  1.5522e-01,
         -2.9989e-01,  5.7053e-01,  9.3016e-01, -2.1168e-01, -3.5144e-02,
          3.7788e-01,  9.0046e-01, -1.1600e+00,  2.9452e-01, -1.4599e-01,
          1.1156e+00, -1.2043e+00, -5.

In [60]:
## Creating ReLU layer
relu = nn.ReLU()

In [61]:
hidden_relu = relu(hidden)

hidden_relu.shape

torch.Size([1, 128])

In [62]:
hidden.shape

torch.Size([1, 128])

In [63]:
hidden[0][:20]

tensor([-0.4209,  0.6806, -0.4231, -0.6120, -0.2436, -0.1003, -0.2293,  0.3991,
         0.5040,  0.2054, -0.0904, -0.0328,  0.0740, -0.2537,  0.6423, -0.3497,
        -0.0225, -0.8998, -0.9063,  0.0354], grad_fn=<SliceBackward0>)

In [64]:
hidden_relu[0][:20] #max (0,x)

tensor([0.0000, 0.6806, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3991, 0.5040,
        0.2054, 0.0000, 0.0000, 0.0740, 0.0000, 0.6423, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0354], grad_fn=<SliceBackward0>)

In [65]:
#output layer
vocab_size = 23206

In [66]:
output_layer = nn.Linear(
    in_features=128,
    out_features=vocab_size
)

In [67]:
output_layer.weight.shape

torch.Size([23206, 128])

In [68]:
output_layer.bias.shape

torch.Size([23206])

In [69]:
scores = output_layer(hidden_relu)

scores.shape

torch.Size([1, 23206])

In [70]:
softmax = nn.Softmax(dim=1)

In [71]:
probs = softmax(scores)

probs.shape

torch.Size([1, 23206])

In [72]:
probs.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [73]:
torch.argmax(probs)

tensor(16463)

In [74]:
loss_fn = nn.CrossEntropyLoss()

In [75]:
target = torch.tensor([1741])

In [77]:
loss = loss_fn(
    scores,
    target
)

loss
#we use raw logits for loss function and not probabilities because the probability computation happens internally in more efficient way

tensor(10.5286, grad_fn=<NllLossBackward0>)


## 1. Data Preparation

### Raw Dataset

```text
movies_metadata.csv
Shape: (45,466, 24)
```

Selected columns:

```text
title | overview | genres
```

Removed rows with missing overviews:

```text
Final Shape: (44,512, 3)
```

---

## 2. Corpus Creation

Example overview:

```text
Led by Woody, Andy's toys live happily...
```

After preprocessing:

```text
['led', 'by', 'woody', 'andys', 'toys', 'live', ...]
```

Each movie overview became a list of tokens.

Result:

```text
44,512 tokenized documents
```

---

## 3. Vocabulary Creation

Counted word frequencies across the entire corpus.

Initial vocabulary:

```text
88,932 unique words
```

Removed rare words appearing ≤ 5 times.

Final vocabulary:

```text
23,206 unique words
```

Created mappings:

```python
word_to_id
id_to_word
```

Example:

```text
led     → 0
by      → 1
woody   → 2
andys   → 3
```

---

## 4. Convert Words to IDs

Example sentence:

```text
['led', 'by', 'woody', 'andys']
```

became:

```text
[0, 1, 2, 3]
```

Neural networks work with numbers, not text.

---

## 5. Create Training Pairs

Using context window = 1.

Example:

```text
[0, 1, 2, 3]
```

Generated:

```text
(0,1)
(1,0)
(1,2)
(2,1)
(2,3)
(3,2)
```

Meaning:

```text
Input Word  → Context Word
```

Result:

```text
4,579,326 training pairs
```

---

## 6. Convert to PyTorch Tensors

Created:

```python
X_tensor
y_tensor
```

Shape:

```text
(4,579,326,)
```

These tensors will be used for training.

---

# Neural Network Architecture

## 1. Embedding Layer

```python
nn.Embedding(
    num_embeddings=23206,
    embedding_dim=100
)
```

Creates:

```text
Embedding Matrix

23206 × 100
```

Each word gets a trainable 100-dimensional vector.

Example:

```text
Word ID 523
↓
[0.66, 0.41, -1.98, ...]
```

At this stage, embeddings are random.

---

## 2. Hidden Linear Layer

```python
nn.Linear(100, 128)
```

Transforms:

```text
100 features
↓
128 features
```

Parameters:

```text
Weights: 128 × 100
Biases: 128
```

---

## 3. ReLU Activation

```python
nn.ReLU()
```

Applies:

```text
ReLU(x) = max(0, x)
```

Example:

```text
Before:
[-0.48, 0.21, -0.89]

After:
[0.00, 0.21, 0.00]
```

Purpose:

```text
Introduce non-linearity
```

No trainable parameters.

---

## 4. Output Layer

```python
nn.Linear(128, 23206)
```

Transforms:

```text
128 hidden features
↓
23206 scores
```

One score for every word in the vocabulary.

Parameters:

```text
Weights: 23206 × 128
Biases: 23206
```

---

## 5. Softmax

Converts scores into probabilities.

Example:

```text
Scores:
[2, 1, 0]

↓

Probabilities:
[0.67, 0.24, 0.09]
```

Properties:

```text
All probabilities ≥ 0
Sum = 1
```

---

## Complete Forward Pass

```text
Input Word ID

↓

Embedding
(23206 × 100)

↓

100-D Embedding Vector

↓

Linear
(100 → 128)

↓

ReLU

↓

Linear
(128 → 23206)

↓

Logits
(23206 scores)

↓

Softmax

↓

Probability of every word
```

---

## Loss Function

Used:

```python
nn.CrossEntropyLoss()
```

Derived from:

```text
Maximum Likelihood Estimation (MLE)
↓
Log Likelihood
↓
Negative Log Likelihood
```

For multiclass classification:

```text
Loss = -log(
Probability of Correct Word
)
```

Low loss:

```text
Model assigns high probability
to the correct context word.
```

High loss:

```text
Model assigns low probability
to the correct context word.
```

---

## Current Status

✅ Data prepared

✅ Vocabulary built

✅ Training pairs generated

✅ Tensors created

✅ Neural network architecture built

✅ Forward pass completed

✅ Softmax probabilities computed

✅ Cross Entropy Loss computed

⏳ Next Step: Backpropagation (`loss.backward()`)

⏳ Then: Weight Updates

⏳ Then: Full Training Loop

⏳ Then: Extract and evaluate learned embeddings

In [78]:
loss_fn = nn.CrossEntropyLoss()

target = torch.tensor([1741])

loss = loss_fn(scores, target)

loss

tensor(10.5286, grad_fn=<NllLossBackward0>)

In [79]:
loss.backward()

In [80]:
embedding.weight.grad.shape

torch.Size([23206, 100])

In [81]:
#Let's verify which emnedding rows got gradients
embedding.weight.grad.abs().sum(dim=1)

tensor([0., 0., 0.,  ..., 0., 0., 0.])

In [82]:
torch.nonzero(
    embedding.weight.grad.abs().sum(dim=1)
)

tensor([[523]])

In [83]:
#Expected because:
#Only the embedding vector for word 523
#participated in this forward pass.

In [84]:
embedding.weight.grad[523][:10]

tensor([ 0.0050,  0.0088,  0.0295,  0.0279, -0.0339, -0.0459, -0.0122,  0.0163,
         0.0100, -0.0143])

In [85]:
#Positive gradient
#→ decrease parameter

#Negative gradient
#→ increase parameter

In [86]:
embedding.weight[523][:5]

tensor([ 1.3168,  1.0903, -0.2130, -0.3454, -0.0685], grad_fn=<SliceBackward0>)

In [87]:
##Applying gradient descent manually
learning_rate = 0.01

with torch.no_grad():
    embedding.weight -= learning_rate * embedding.weight.grad

In [88]:
embedding.weight[523][:5]

tensor([ 1.3168,  1.0902, -0.2133, -0.3456, -0.0681], grad_fn=<SliceBackward0>)

In [ ]:
#what we saw above is done by optimizer.step()